# CatBoost

A gradient-boosted tree classifier with native categorical processing compares `application_only` and `application_history` features from the shared preparation cache. Both variants use the same applicant splits.

Validation ROC-AUC selects the model; classification metrics use a fixed 0.5 threshold. Holdout and competition data are not evaluated.


In [10]:
%pip install -q pandas numpy "scikit-learn>=1.3" catboost

from pathlib import Path
import sys

REPO_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "scripts" / "project_data.py").is_file()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Notebook working directory must be inside this repository.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import gc
from time import perf_counter
import numpy as np
import pandas as pd
from IPython.display import display
from scripts.project_data import FEATURE_SETS, load_prepared_data, get_partitions
from scripts.experiment_utils import (
    validation_metrics, split_fingerprint, feature_schema_fingerprint,
    package_versions, save_run_record,
)


FEATURE_SET = "application_history"  # or "application_history"; restart/rerun after changing
SPLIT_STRATEGY = "stratified"      # Alternative: "random"
RANDOM_STATE = 42
CLASSIFICATION_THRESHOLD = 0.5
ACTIVE_PARTITIONS = ("train", "validation")

Note: you may need to restart the kernel to use updated packages.


## 1. Feature set and shared splits

The prepared cache contains application-only and application-plus-history features with identical applicant assignments. This notebook selects training and validation rows for `FEATURE_SET`, records the split fingerprint, and releases the full cache bundle.


In [11]:
preparation_started = perf_counter()
prepared_data = load_prepared_data()
run_split_fingerprint = split_fingerprint(
    prepared_data["split_ids"][SPLIT_STRATEGY], prepared_data["labels"],
)
data = get_partitions(
    prepared_data, FEATURE_SET, SPLIT_STRATEGY, partitions=ACTIVE_PARTITIONS,
)
raw_inputs = data["raw_inputs"]
labels = data["labels"]
applicant_ids = data["applicant_ids"]
audit_by_partition = data["audit_by_partition"]
numeric_features = data["numeric_features"]
categorical_features = data["categorical_features"]
run_feature_schema = feature_schema_fingerprint(raw_inputs["train"].columns)
prepared_schema_version = prepared_data["schema_version"]
raw_feature_count = raw_inputs["train"].shape[1]
run_row_counts = {part: len(raw_inputs[part]) for part in ACTIVE_PARTITIONS}
run_positive_rates = {part: float(labels[part].mean()) for part in ACTIVE_PARTITIONS}
del prepared_data, data
gc.collect()
assert set(raw_inputs) == {"train", "validation"}
display(pd.DataFrame([
    {"feature_set": FEATURE_SET, "partition": part, "rows": len(frame), "raw_features": frame.shape[1]}
    for part, frame in raw_inputs.items()
]))
print("Protocol:", SPLIT_STRATEGY)

,feature_set,partition,rows,raw_features
0,application_history,train,215257,303
1,application_history,validation,46127,303


Protocol: stratified


## 2. CatBoost preprocessing

Numeric features retain their original scales and missing values. Categorical features are converted to strings, with missing values represented by `__MISSING__`; unseen strings remain available for native processing. No manual target encoding is applied. [Categorical-value requirements](https://catboost.ai/docs/en/concepts/faq).


In [12]:
def prepare_catboost(frame, categorical_features):
    prepared = frame.copy()
    for column in categorical_features:
        prepared[column] = prepared[column].astype("object").fillna("__MISSING__").astype(str)
    return prepared


catboost_cat_features = categorical_features
catboost_inputs = {
    part: prepare_catboost(raw_inputs[part], categorical_features)
    for part in ACTIVE_PARTITIONS
}

## 3. Input alignment

The DataFrames retain applicant-ID alignment, numeric values are unchanged, and every categorical value is a string. Raw frames are released before fitting.

In [13]:
for part in ACTIVE_PARTITIONS:
    frame = catboost_inputs[part]
    raw = raw_inputs[part]
    assert frame.index.equals(raw.index) and frame.columns.equals(raw.columns)
    np.testing.assert_array_equal(frame.index.to_numpy(), applicant_ids[part])
    assert not np.isinf(frame[numeric_features].to_numpy()).any()
    pd.testing.assert_frame_equal(frame[numeric_features], raw[numeric_features])
    for column in catboost_cat_features:
        assert frame[column].notna().all()
        assert frame[column].map(lambda value: isinstance(value, str)).all()
    assert frame.index.equals(labels[part].index)
display(pd.DataFrame([
    {"partition": part, "rows": len(catboost_inputs[part]), "features": catboost_inputs[part].shape[1]}
    for part in ACTIVE_PARTITIONS
]))
del raw_inputs, raw, frame
gc.collect()
preparation_seconds = perf_counter() - preparation_started
print(f"Preparation finished in {preparation_seconds:.1f}s. No training yet.")

,partition,rows,features
0,train,215257,303
1,validation,46127,303


Preparation finished in 9.6s. No training yet.


## 4. Model configuration

Depth-6 trees use learning rate 0.05, at most 1,000 iterations, L2 regularization 3, and the Logloss training objective.

Validation AUC controls early stopping with patience of 75 iterations, and the best model is retained. Computation uses four CPU threads and seed 42, without class weighting or CatBoost's auxiliary training files. [Classifier API](https://catboost.ai/docs/en/concepts/python-reference_catboostclassifier).


In [14]:
try:
    from catboost import CatBoostClassifier
except ImportError as exc:
    raise ImportError("Run %pip install catboost in this kernel, then restart/rerun.") from exc

CONFIGURATION_NAME = "starter_v1"
CATBOOST_CONFIG = {
    "iterations": 1000, "depth": 6, "learning_rate": 0.05, "l2_leaf_reg": 3.0,
    "loss_function": "Logloss", "eval_metric": "AUC", "random_seed": RANDOM_STATE,
    "thread_count": 4, "task_type": "CPU", "allow_writing_files": False,
}
EARLY_STOPPING_ROUNDS = 75
print(CATBOOST_CONFIG)

{'iterations': 1000, 'depth': 6, 'learning_rate': 0.05, 'l2_leaf_reg': 3.0, 'loss_function': 'Logloss', 'eval_metric': 'AUC', 'random_seed': 42, 'thread_count': 4, 'task_type': 'CPU', 'allow_writing_files': False}


## 5. Training

The classifier fits training rows only. Validation ROC-AUC controls early stopping and selects the retained model; labeled holdout and competition rows are excluded.


In [15]:
model = CatBoostClassifier(**CATBOOST_CONFIG)
training_started = perf_counter()
model.fit(
    catboost_inputs["train"], labels["train"],
    cat_features=catboost_cat_features,
    eval_set=(catboost_inputs["validation"], labels["validation"]),
    early_stopping_rounds=EARLY_STOPPING_ROUNDS, use_best_model=True, verbose=100,
)
training_seconds = perf_counter() - training_started
best_iteration = int(model.get_best_iteration()) + 1  # CatBoost's index is zero-based
prediction_started = perf_counter()
validation_probability = model.predict_proba(catboost_inputs["validation"])[:, 1]
prediction_seconds = perf_counter() - prediction_started
run_parameters = {
    **CATBOOST_CONFIG, "early_stopping_rounds": EARLY_STOPPING_ROUNDS, "use_best_model": True,
}
training_history = model.get_evals_result()
model_feature_count = catboost_inputs["train"].shape[1]
print(f"Selected trees: {best_iteration}; training time: {training_seconds:.1f}s")

0:	test: 0.5362630	best: 0.5362630 (0)	total: 760ms	remaining: 12m 39s
100:	test: 0.7679441	best: 0.7679441 (100)	total: 1m 6s	remaining: 9m 53s
200:	test: 0.7759181	best: 0.7759296 (199)	total: 2m 11s	remaining: 8m 42s
300:	test: 0.7803202	best: 0.7803202 (300)	total: 3m 15s	remaining: 7m 33s
400:	test: 0.7827215	best: 0.7827341 (399)	total: 4m 17s	remaining: 6m 25s
500:	test: 0.7842377	best: 0.7842377 (500)	total: 5m 21s	remaining: 5m 19s
600:	test: 0.7851920	best: 0.7851924 (599)	total: 6m 24s	remaining: 4m 15s
700:	test: 0.7861912	best: 0.7861912 (700)	total: 7m 27s	remaining: 3m 10s
800:	test: 0.7867470	best: 0.7867496 (798)	total: 8m 29s	remaining: 2m 6s
900:	test: 0.7870971	best: 0.7871033 (899)	total: 9m 32s	remaining: 1m 2s
999:	test: 0.7875461	best: 0.7875625 (997)	total: 10m 33s	remaining: 0us

bestTest = 0.7875624794
bestIteration = 997

Shrink model to first 998 iterations.
Selected trees: 998; training time: 640.2s


## 6. Validation metrics

ROC-AUC is the primary metric, complemented by average precision, log loss, accuracy, precision, recall, and F1. Classification metrics use a fixed 0.5 threshold. The confusion matrix places actual labels in rows and predicted labels in columns.

A run record in `results/metrics/` stores metrics, configuration, timings, learning history, library versions, and split/feature fingerprints for cross-model comparison.


In [16]:
metrics = validation_metrics(
    labels["validation"], validation_probability, threshold=CLASSIFICATION_THRESHOLD,
)
display(pd.DataFrame([
    {"metric": key, "value": value}
    for key, value in metrics.items() if key not in {"confusion_matrix", "threshold"}
]))
counts = metrics["confusion_matrix"]
display(pd.DataFrame(
    [[counts["tn"], counts["fp"]], [counts["fn"], counts["tp"]]],
    index=pd.Index(["0", "1"], name="Actual"),
    columns=pd.Index(["0", "1"], name="Predicted"),
))
run_record = {
    "model": "catboost", "feature_set": FEATURE_SET, "split_strategy": SPLIT_STRATEGY,
    "configuration": CONFIGURATION_NAME, "seed": RANDOM_STATE, "evaluation_partition": "validation",
    "optimizing_metric": "roc_auc", "split_fingerprint": run_split_fingerprint,
    "feature_schema_fingerprint": run_feature_schema, "prepared_schema_version": prepared_schema_version,
    "raw_feature_count": raw_feature_count, "model_feature_count": model_feature_count,
    "rows": run_row_counts, "positive_rates": run_positive_rates, "parameters": run_parameters,
    "selected_iteration": best_iteration, "preparation_seconds": preparation_seconds,
    "training_seconds": training_seconds, "prediction_seconds": prediction_seconds,
    "metrics": metrics, "training_history": training_history, "versions": package_versions("catboost"),
    "holdout_evaluated": False, "class_weighting": "none", "threshold_tuned": False,
}
result_path = save_run_record(run_record)
print(f"Saved validation results: {result_path}")
print("Holdout not evaluated. No Kaggle submission created.")

,metric,value
0,roc_auc,0.787562
1,average_precision,0.285219
2,accuracy,0.920242
3,f1,0.085508
4,precision,0.575251
5,recall,0.046187
6,log_loss,0.236761


Predicted,0,1
Actual,,
0,42276,127
1,3552,172


Saved validation results: C:\Users\CRNeu\Documents\Data4401\Data4401-Project2\results\metrics\catboost_application_history_stratified_starter_v1_20261006T011403951338Z_db5ea82e.json
Holdout not evaluated. No Kaggle submission created.


## 7. Applicant-level validation predictions

The existing validation probabilities are saved to `results/predictions/` as a CSV with matching run metadata. Each row contains applicant ID, actual target, predicted probability and label, error type (TN/FP/FN/TP), gender, age, age group, and run ID.

Metadata links predictions to the metric record, split, configuration, and threshold. These artifacts support downstream error and fairness analysis without refitting. Only validation rows are exported; missing audit categories remain `UNKNOWN` and missing numeric ages remain blank.


In [ ]:
from scripts import experiment_utils as _prediction_utils

PREDICTION_EXPORT_DIR = None  # Default: results/predictions/ in this repo.
prediction_csv_path, prediction_metadata_path = _prediction_utils.export_validation_predictions(
    applicant_ids=applicant_ids["validation"],
    labels=labels["validation"],
    probabilities=validation_probability,
    audit=audit_by_partition["validation"],
    result_path=result_path,
    directory=PREDICTION_EXPORT_DIR,
)
print(f"Validation predictions: {prediction_csv_path}")
print(f"Run metadata: {prediction_metadata_path}")
print(f"CSV: {prediction_csv_path.stat().st_size / 1024**2:.2f} MiB; metadata: {prediction_metadata_path.stat().st_size / 1024:.1f} KiB")
print(f"Rows: {len(labels['validation']):,}; feature set: {run_record['feature_set']}")
print("Validation-only predictions exported; no model refitted.")
